In [3]:
!pip install transformers accelerate -q

In [4]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


In [7]:
text = """
The mountains stood tall against the morning sky, their peaks dusted with
fresh snow that sparkled in the early light. A gentle breeze moved through
the pine trees, carrying with it the crisp scent of winter. Down in the
valley, a small village was waking up, smoke curling from chimneys as
families prepared for the day ahead. Children laughed as they walked to
school, their boots crunching against the frosty ground. The river that
ran beside the village was partially frozen, its edges lined with delicate
ice patterns, while the center still flowed steadily, refusing to be
completely still even in the coldest months. Above, a hawk circled slowly,
scanning the fields below for any sign of movement. Life in the mountains
moved at its own pace, unhurried and steady, shaped by the rhythm of the
seasons rather than the rush of the modern world. As the sun rose higher,
the snow began to glisten even brighter, turning the entire landscape into
something that looked almost magical, like a scene from an old painting
brought to life. The villagers had lived this way for generations, finding
comfort in routine and strength in their connection to the land around them.
"""

with open("mydata.txt", "w") as f:
    f.write(text)

print(text[:500])
print("\nTotal characters:", len(text))


The mountains stood tall against the morning sky, their peaks dusted with
fresh snow that sparkled in the early light. A gentle breeze moved through
the pine trees, carrying with it the crisp scent of winter. Down in the
valley, a small village was waking up, smoke curling from chimneys as
families prepared for the day ahead. Children laughed as they walked to
school, their boots crunching against the frosty ground. The river that
ran beside the village was partially frozen, its edges lined wit

Total characters: 1175


In [10]:
import torch
from torch.utils.data import Dataset

# Read and tokenize your text
with open("mydata.txt", "r") as f:
    raw_text = f.read()

tokenized_text = tokenizer.encode(raw_text)

block_size = 64

class SimpleDataset(Dataset):
    def __init__(self, tokenized_text, block_size):
        self.examples = []
        for i in range(0, len(tokenized_text) - block_size + 1, block_size):
            self.examples.append(tokenized_text[i:i + block_size])

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return torch.tensor(self.examples[idx], dtype=torch.long)

dataset = SimpleDataset(tokenized_text, block_size)
print("Number of training examples:", len(dataset))

Number of training examples: 4


In [12]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    logging_steps=1,
)

print("Training config ready!")

Training config ready!


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,4.203218
2,4.160555
3,3.894855
4,3.237792
5,3.122609
6,3.543749


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=3.693796435991923, metrics={'train_runtime': 17.2517, 'train_samples_per_second': 0.696, 'train_steps_per_second': 0.348, 'total_flos': 391938048000.0, 'train_loss': 3.693796435991923, 'epoch': 3.0})

In [14]:
model.save_pretrained("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [16]:
prompt = "The mountains"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

print("=== GREEDY SEARCH ===")
output = model.generate(input_ids, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

print("\n=== BEAM SEARCH ===")
output = model.generate(input_ids, max_new_tokens=50, num_beams=5, no_repeat_ngram_size=2)
print(tokenizer.decode(output[0], skip_special_tokens=True))

print("\n=== TOP-K SAMPLING ===")
output = model.generate(input_ids, max_new_tokens=50, do_sample=True, top_k=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

print("\n=== TOP-P (NUCLEUS) SAMPLING ===")
output = model.generate(input_ids, max_new_tokens=50, do_sample=True, top_p=0.92, top_k=0)
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== GREEDY SEARCH ===


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The mountains of the mountains of the mountains of the mountains of the world are the most beautiful. The mountains of the world are the most beautiful. The mountains of the mountains of the world are the most beautiful. The mountains of the world are the most beautiful.

=== BEAM SEARCH ===


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The mountains in the center of the city were filled with people of all ages, from all walks of life, all over the world. The city was full of people, people who had lived for centuries, and lived in harmony with each other. It was a

=== TOP-K SAMPLING ===


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The mountains ahead of him and with the great waves. heaving the deep black with his magic, he stood atop a rock, a light in his fingertips.

In the distance, was still a stream of wind. His mind could feel his breathing tighten

=== TOP-P (NUCLEUS) SAMPLING ===
The mountains are haunted by a sea, showing the hundreds of thousands of good souls who were living in its wisdom, so that they accepted the rest of their lives. A rich oak forest springs through it, just as the wind was blowing. No way was it
